In [3]:
import os
import re
import time
import random
import pickle
from collections import Counter

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix
from tqdm.auto import tqdm

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(42)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE}")

Device: cpu


/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
# ---------------------------------------------------------------------------
# Paths
# ---------------------------------------------------------------------------
DATA_DIR = "liar2"          # folder containing train.csv, valid.csv, test.csv
TRAIN_CSV = os.path.join(DATA_DIR, "train.csv")
VALID_CSV = os.path.join(DATA_DIR, "valid.csv")
TEST_CSV = os.path.join(DATA_DIR, "test.csv")

CHECKPOINT_DIR = "checkpoints"
VOCAB_PATH = os.path.join(CHECKPOINT_DIR, "vocabs.pkl")
BEST_MODEL_PATH = os.path.join(CHECKPOINT_DIR, "fdhn_best.pt")
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

# ---------------------------------------------------------------------------
# Dataset / label settings
# ---------------------------------------------------------------------------
NUM_CLASSES = 6
LABEL_NAMES = ["pants-fire", "false", "barely-true", "half-true", "mostly-true", "true"]

# Actual LIAR2 CSV column names for the six credibility-history features
# (f9 - f14 in the paper's Table 2). Note: the paper's text calls f12
# 'barely_true_counts', but the actual released CSV column is 'mostly_false_counts'.
NUMERICAL_COLUMNS = [
    "true_counts", "mostly_true_counts", "half_true_counts",
    "mostly_false_counts", "false_counts", "pants_on_fire_counts",
]

# Textual context columns (f2, f3, f4, f6, f7, f15) - concatenated into one string
CONTEXT_COLUMNS = ["date", "subject", "speaker", "speaker_description", "state_info", "context"]

STATEMENT_COLUMN = "statement"
JUSTIFICATION_COLUMN = "justification"
LABEL_COLUMN = "label"

# ---------------------------------------------------------------------------
# Tokenization / sequence lengths
# ---------------------------------------------------------------------------
MAX_STATEMENT_LEN = 40        # avg statement length is ~17.7 tokens (Table 1)
MAX_CONTEXT_LEN = 60
MAX_JUSTIFICATION_LEN = 120   # avg justification length is ~94.4 tokens (Table 1)
MIN_WORD_FREQ = 2             # words appearing fewer times than this become <UNK>

PAD_TOKEN, PAD_IDX = "<PAD>", 0
UNK_TOKEN, UNK_IDX = "<UNK>", 1

# ---------------------------------------------------------------------------
# Model hyperparameters (Section V-A)
# ---------------------------------------------------------------------------
EMBED_DIM = 128
KERNEL_SIZES = (3, 4, 5)
NUM_FILTERS = 128
DROPOUT = 0.5

NUMERICAL_FEATURE_DIM = 64    # hidden size the 6 raw numerical features get projected to
CNN_OUT_CHANNELS = 32         # CNNBiLSTM's CNN layer output channels, kernel_size=1
LSTM_HIDDEN = 64              # per-direction hidden size; Bi-LSTM -> 128-dim output

USE_JUSTIFICATION = True      # Section V-D: LIAR2 experiments add a 5th TextCNN branch

# ---------------------------------------------------------------------------
# Training hyperparameters
# ---------------------------------------------------------------------------
BATCH_SIZE = 64
LEARNING_RATE = 1e-3
NUM_EPOCHS = 10               # paper: "all model training phases completed within 10 epochs"
SEED = 42


In [5]:
def simple_tokenize(text):
    """Lowercase, keep only alphanumerics, split on whitespace."""
    if not isinstance(text, str):
        return []
    text = text.lower()
    text = re.sub(r"[^a-z0-9\s]", " ", text)
    return text.split()


def build_vocab(token_lists, min_freq=MIN_WORD_FREQ):
    counter = Counter()
    for tokens in token_lists:
        counter.update(tokens)
    vocab = {PAD_TOKEN: PAD_IDX, UNK_TOKEN: UNK_IDX}
    for word, freq in counter.items():
        if freq >= min_freq:
            vocab[word] = len(vocab)
    return vocab


def encode_tokens(tokens, vocab, max_len):
    ids = [vocab.get(t, UNK_IDX) for t in tokens[:max_len]]
    if len(ids) < max_len:
        ids = ids + [PAD_IDX] * (max_len - len(ids))
    return ids


def build_context_text(row):
    parts = []
    for col in CONTEXT_COLUMNS:
        val = row.get(col, "")
        if pd.notna(val):
            parts.append(str(val))
    return " ".join(parts)


def fit_vocabs_and_scaler(train_df):
    """Fit everything on the TRAINING split only, to avoid leakage."""
    statement_tokens = [simple_tokenize(s) for s in train_df[STATEMENT_COLUMN]]
    context_tokens = [simple_tokenize(build_context_text(row)) for _, row in train_df.iterrows()]
    justification_tokens = [
        simple_tokenize(s) for s in train_df.get(JUSTIFICATION_COLUMN, pd.Series([""] * len(train_df)))
    ]

    statement_vocab = build_vocab(statement_tokens)
    context_vocab = build_vocab(context_tokens)
    justification_vocab = build_vocab(justification_tokens)

    numerical = train_df[NUMERICAL_COLUMNS].fillna(0).astype(float).values
    mean = numerical.mean(axis=0)
    std = numerical.std(axis=0)
    std[std == 0] = 1.0

    return {
        "statement_vocab": statement_vocab,
        "context_vocab": context_vocab,
        "justification_vocab": justification_vocab,
        "numerical_mean": mean,
        "numerical_std": std,
    }


def encode_row(row, vocabs):
    """Encode a single dataframe row into model-ready arrays."""
    statement_ids = encode_tokens(simple_tokenize(row[STATEMENT_COLUMN]), vocabs["statement_vocab"], MAX_STATEMENT_LEN)
    context_ids = encode_tokens(simple_tokenize(build_context_text(row)), vocabs["context_vocab"], MAX_CONTEXT_LEN)
    justification_ids = encode_tokens(
        simple_tokenize(row.get(JUSTIFICATION_COLUMN, "")), vocabs["justification_vocab"], MAX_JUSTIFICATION_LEN
    )

    raw_numerical = np.array([float(row.get(c, 0) or 0) for c in NUMERICAL_COLUMNS], dtype=np.float32)
    numerical = (raw_numerical - vocabs["numerical_mean"]) / vocabs["numerical_std"]

    label = int(row[LABEL_COLUMN])

    return {
        "statement": np.array(statement_ids, dtype=np.int64),
        "context": np.array(context_ids, dtype=np.int64),
        "justification": np.array(justification_ids, dtype=np.int64),
        "numerical": numerical.astype(np.float32),
        "label": label,
    }

In [6]:
print(f"Loading training data from {TRAIN_CSV} ...")
train_df = pd.read_csv(TRAIN_CSV)
print(f"  {len(train_df)} rows")

print("Fitting vocabularies and scaler on training split ...")
vocabs = fit_vocabs_and_scaler(train_df)
print(f"  statement vocab size:     {len(vocabs['statement_vocab'])}")
print(f"  context vocab size:       {len(vocabs['context_vocab'])}")
print(f"  justification vocab size: {len(vocabs['justification_vocab'])}")

with open(VOCAB_PATH, "wb") as f:
    pickle.dump(vocabs, f)
print(f"Saved vocabularies to {VOCAB_PATH}")

Loading training data from liar2/train.csv ...
  18369 rows
Fitting vocabularies and scaler on training split ...
  statement vocab size:     10046
  context vocab size:       8226
  justification vocab size: 21756
Saved vocabularies to checkpoints/vocabs.pkl


In [7]:
from torch.utils.data import Dataset, DataLoader

class LiarDataset(Dataset):
    def __init__(self, csv_path, vocabs):
        self.df = pd.read_csv(csv_path)
        self.vocabs = vocabs

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        encoded = encode_row(row, self.vocabs)
        return {
            "statement": torch.tensor(encoded["statement"], dtype=torch.long),
            "context": torch.tensor(encoded["context"], dtype=torch.long),
            "justification": torch.tensor(encoded["justification"], dtype=torch.long),
            "numerical": torch.tensor(encoded["numerical"], dtype=torch.float32),
            "label": torch.tensor(encoded["label"], dtype=torch.long),
        }


train_ds = LiarDataset(TRAIN_CSV, vocabs)
valid_ds = LiarDataset(VALID_CSV, vocabs)
test_ds = LiarDataset(TEST_CSV, vocabs)


In [ ]:
from torch.utils.data import Dataset, DataLoader

class LiarDataset(Dataset):
    def __init__(self, csv_path, vocabs):
        self.df = pd.read_csv(csv_path)
        self.vocabs = vocabs

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        encoded = encode_row(row, self.vocabs)
        return {
            "statement": torch.tensor(encoded["statement"], dtype=torch.long),
            "context": torch.tensor(encoded["context"], dtype=torch.long),
            "justification": torch.tensor(encoded["justification"], dtype=torch.long),
            "numerical": torch.tensor(encoded["numerical"], dtype=torch.float32),
            "label": torch.tensor(encoded["label"], dtype=torch.long),
        }


train_ds = LiarDataset(TRAIN_CSV, vocabs)
valid_ds = LiarDataset(VALID_CSV, vocabs)
test_ds = LiarDataset(TEST_CSV, vocabs)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
valid_loader = DataLoader(valid_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

print(f"Train batches: {len(train_loader)} | Valid batches: {len(valid_loader)} | Test batches: {len(test_loader)}")

NameError: name 'Dataset' is not defined